In [1]:
# activity data
import pandas as pd
import numpy as np
import os 
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler

from sklearn.ensemble import RandomForestClassifier,GradientBoostingClassifier
from sklearn.gaussian_process import GaussianProcessClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB

from sklearn.metrics import f1_score,confusion_matrix,precision_score,recall_score
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler

from imblearn.over_sampling import RandomOverSampler

import matplotlib.pyplot as plt
import seaborn as sns
import shap

from utils.data_utils import correct_col_type,gen_date_col,transform_category_to_counts,min_max_perpatient

In [2]:
import os
import pandas as pd
# Please change the path with the path of your dataset
DPATH = '../Dataset/'

# Read all tables into data_dict

files = os.listdir(DPATH)
data_dict = {}
summaries = {}
for f in files:
    if 'csv' not in f:
        continue
    print(f)
    fpth = os.path.join(DPATH,f)
    df = pd.read_csv(fpth)

    for col in df.columns:
        df[col] = correct_col_type(df,col)
    if 'date' in df.columns:
        df = df.rename(columns={'date':'timestamp'})
                
    fname = f.split('.')[0]
    data_dict[fname] = df

Activity.csv
Demographics.csv
Labels.csv
Physiology.csv
Sleep.csv


In [3]:
act_df1 = gen_date_col(data_dict['Activity'],tcol='timestamp')
act_df1

,patient_id,location_name,timestamp,date
0,0697d,Fridge Door,2019-06-28 13:03:29,2019-06-28
1,0697d,Kitchen,2019-06-28 13:11:44,2019-06-28
2,0697d,Front Door,2019-06-28 13:13:50,2019-06-28
3,0697d,Bedroom,2019-06-28 13:13:53,2019-06-28
4,0697d,Fridge Door,2019-06-28 13:14:09,2019-06-28
...,...,...,...,...
1030554,fd100,Hallway,2019-06-30 23:48:50,2019-06-30
1030555,fd100,Lounge,2019-06-30 23:49:40,2019-06-30
1030556,fd100,Kitchen,2019-06-30 23:50:02,2019-06-30
1030557,fd100,Front Door,2019-06-30 23:51:28,2019-06-30


# activity location transition entropy per person per day

In [4]:
sleep_id_array = act_df1.patient_id.unique()
sleep_id_array

['0697d', '099bc', '0cda9', '0d5ef', '0efe8', ..., 'e8a78', 'ec812', 'eca1f', 'f220c', 'fd100']
Length: 56
Categories (56, object): ['0697d', '099bc', '0cda9', '0d5ef', ..., 'ec812', 'eca1f', 'f220c', 'fd100']

In [15]:
import pandas as pd
import datetime

entropy_df_list = []
for id_pick in sleep_id_array:
    print(id_pick) 
    act_df_person = act_df1[act_df1['patient_id'] == id_pick] 
    
    date_array = act_df_person.date.unique()
    entropy_rate_list = []
    date_list = []  
    
    for target_date in date_array:
        date_list.append(target_date)
        
        # in each day
        filtered_df = act_df_person[act_df_person['date'] == target_date]        
        filtered_df['timestamp'] = pd.to_datetime(filtered_df['timestamp'])
        filtered_df = filtered_df.sort_values('timestamp').reset_index(drop=True)
        
        filtered_df['timestamp'] = pd.to_datetime(filtered_df['timestamp'])
        
        # Sort by timestamp
        filtered_df = filtered_df.sort_values('timestamp').reset_index(drop=True)
        
##################### calculate bout duration ##########################
        # Identify chunks of consecutive same state
        filtered_df['location_change'] = (filtered_df['location_name'] != filtered_df['location_name'].shift()).cumsum()
        
        # Group by each chunk
        chunk_durations = filtered_df.groupby(['location_change', 'location_name']).agg(
            start_time=('timestamp', 'first'),
            end_time=('timestamp', 'last')
        ).reset_index()
        chunk_durations = chunk_durations.dropna(subset=['start_time']).reset_index()
        
        # Calculate duration in minutes per chunk
        chunk_durations['duration_min'] = (chunk_durations['end_time'] - chunk_durations['start_time']).dt.total_seconds() / 60
        
        
        # Your sequence of locations
        locations = chunk_durations.location_name
        
        # Create a list of unique locations (states)
        unique_locations = sorted(set(locations))
        # if there is no transition
        if len(unique_locations) <= 1:  
            entropy_rate = np.NAN
        else:    
            # Initialize the transition matrix with zeros
            transition_matrix = pd.DataFrame(0, index=unique_locations, columns=unique_locations)
            
            # Count the transitions
            for i in range(len(locations) - 1):
                from_location = locations[i]
                to_location = locations[i + 1]
                transition_matrix.at[from_location, to_location] += 1
            
            # Convert counts to probabilities by dividing each row by its sum
            transition_matrix = transition_matrix.div(transition_matrix.sum(axis=1), axis=0)
            
            # if the transition matrix contain NAN
            if transition_matrix.isna().values.any():
                entropy_rate = np.NAN
                
            else:      
                # Step 4: Estimate stationary distribution π
                # Method: left eigenvector of P.T corresponding to eigenvalue 1
                # I have checked the result with another function (https://stephens999.github.io/fiveMinuteStats/stationary_distribution.html) the results are the same
                eigvals, eigvecs = np.linalg.eig(transition_matrix.T)
                stat_dist = np.real(eigvecs[:, np.isclose(eigvals, 1)])
                stat_dist = stat_dist[:, 0]
                stat_dist = stat_dist / stat_dist.sum()
                stat_dist = pd.DataFrame({'stat_dist': stat_dist, 'unique_locations': unique_locations}).set_index('unique_locations')       
        
                # Step 5: Compute entropy rate
                P = transition_matrix.copy()
                entropy_rate = 0.0
                for i in unique_locations:
                    for j in unique_locations:
                        if P.at[i, j] > 0:
                            entropy_rate -= stat_dist.at[i,'stat_dist'] * P.at[i, j] * np.log2(P.at[i, j])
        # normalize the original entropy rate by log2( N), N is the number of unique location 
        entropy_rate_normalized = entropy_rate / np.log2(len(unique_locations))
        entropy_rate_list.append(entropy_rate_normalized) 
            
    entropy_person = {
    'patient_id': id_pick,
    'date': date_list,
    'entropy_rate': entropy_rate_list 
    }
        
    entropy_person_df = pd.DataFrame(entropy_person)  
    entropy_df_list.append(entropy_person_df)

0697d
099bc
0cda9
0d5ef
0efe8
0f352
16f4b
1fbe4
201d8
28710
2b131
2f54b
30a32
385de
393cb
3fb61
46286
55cd4
561af
56b6b
65db4
6b29b
714d7
73f7c
76230
7db78
8a835
8d0d4
93c14
95899
96adf
a2849
a380e
a539e
ab47a
b0455
b45c2
b9d58
c5031
c55f8
c5785
c8574
ca44d
d263a
d44d2
d7a46
d8d97
d93d8
e2472
e4959
e87bd
e8a78
ec812
eca1f
f220c
fd100


In [16]:
final_df = pd.concat(entropy_df_list, ignore_index=True)

final_df.to_csv('../output/data_cleaned/activity_entropy_rates.csv', index=False)
final_df

,patient_id,date,entropy_rate
0,0697d,2019-06-28,0.586975
1,0697d,2019-06-29,0.584840
2,0697d,2019-06-30,0.475296
3,099bc,2019-05-15,0.612263
4,099bc,2019-05-16,0.576208
...,...,...,...
2717,f220c,2019-06-30,0.681595
2718,fd100,2019-06-27,0.599583
2719,fd100,2019-06-28,0.654796
2720,fd100,2019-06-29,0.659778


# room specific activity count

In [17]:
act_df = transform_category_to_counts(act_df1,col='location_name',keys=['patient_id','date'])
act_df.to_csv('../output/data_cleaned/activity_location_daily_count.csv',index=False)
act_df 

location_name,patient_id,date,Back Door,Bathroom,Bedroom,Fridge Door,Front Door,Hallway,Kitchen,Lounge
0,0697d,2019-06-28,14,7,24,23,28,40,106,80
1,0697d,2019-06-29,2,11,26,8,23,57,120,117
2,0697d,2019-06-30,4,14,53,0,8,57,119,103
3,099bc,2019-05-15,12,17,31,27,13,32,71,6
4,099bc,2019-05-16,14,42,85,22,6,50,104,9
...,...,...,...,...,...,...,...,...,...,...
2717,f220c,2019-06-30,0,12,61,24,18,32,0,0
2718,fd100,2019-06-27,2,24,48,23,25,47,100,97
2719,fd100,2019-06-28,0,32,91,7,21,58,145,120
2720,fd100,2019-06-29,0,33,56,27,15,61,110,96
